## Install Vertex AI SDK

### Subtask:
Install the `google-cloud-aiplatform` library, which is required to interact with Vertex AI services.


**Reasoning**:
Install the `google-cloud-aiplatform` library using pip as instructed by the subtask.



In [ ]:
import sys
!{sys.executable} -m pip install -U -q google-cloud-aiplatform pandas==2.2.2 seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 45.1 MB/s eta 0:00:00


### Set up Service Account Credentials (Alternative to `auth.authenticate_user()`)

In [1]:
from google import genai
from google.colab import auth

# Authenticate user for Colab environment
auth.authenticate_user()


# Vertex AI Embedding Model
EMBEDDING_MODEL_VERTEX_AI = "gemini-embedding-2-preview"

client = genai.Client(
    vertexai=True, project='demoproject-cbbe7', location='us-central1'
)


ModuleNotFoundError: No module named 'google.colab'

### Define Multimodal Embedding Function using `genai.Client`

In [ ]:
import numpy as np

# Example Data: A mix of text descriptions and image URLs
database = [
    {"id": "doc1", "type": "text", "content": "A golden retriever playing in the park.", "embedding": None},
    {"id": "eiffel", "type": "image", "url": "https://cdn.pixabay.com/photo/2024/09/21/15/07/eiffel-tower-9064240_640.jpg", "embedding": None},
    {"id": "dog", "type": "image", "url": "https://cdn.shopify.com/s/files/1/0086/0795/7054/files/Golden-Retriever.jpg", "embedding": None},
    {"id": "fashion", "type": "image", "url": "https://i.pinimg.com/564x/85/22/34/8522346c05525356198706df30c7ebe0.jpg", "embedding": None}

]

def cosine_similarity(a, b):
    """Calculates the cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Database and cosine_similarity function restored.")

In [ ]:
import requests
from PIL import Image
from io import BytesIO
from google import genai

def get_multimodal_embedding(text=None, image_path=None):
    """Generates a multimodal embedding using the genai.Client with Vertex AI backend."""
    contents = []
    if text:
        contents.append(text)
    if image_path:
        if image_path.startswith('http'):
            response = requests.get(image_path)
            img_bytes = BytesIO(response.content).getvalue()
            # Corrected: Create a Blob for inline_data and pass it to Part constructor
            contents.append(genai.types.Part(inline_data=genai.types.Blob(data=img_bytes, mime_type='image/jpeg')))
        else:
            with open(image_path, 'rb') as f:
                img_bytes = f.read()
            # Corrected: Create a Blob for inline_data and pass it to Part constructor
            contents.append(genai.types.Part(inline_data=genai.types.Blob(data=img_bytes, mime_type='image/jpeg')))

    if not contents:
        raise ValueError("At least one of 'text' or 'image_path' must be provided.")

    response = client.models.embed_content(
        model=EMBEDDING_MODEL_VERTEX_AI,
        contents=contents,
    )
    return response.embeddings[0].values

print("Defined get_multimodal_embedding function using genai.Client.")


### Restore `database` and `cosine_similarity`

### Re-index Database with new Embeddings

In [ ]:
print("Re-indexing database with genai.Client embeddings...")
for item in database:
    if item['type'] == 'text':
        item['embedding'] = get_multimodal_embedding(text=item['content'])
    elif item['type'] == 'image':
        item['embedding'] = get_multimodal_embedding(image_path=item['url'])

    print(f"Re-indexed item ID: {item['id']} \n {item['embedding']}")

print("Database re-indexed successfully.")

### Perform Cross-modal Search

In [ ]:
# Cross-modal Search Example
query_text = "Tell me which pet i should adopt?"
print(f"\nSearching for: '{query_text}'")
query_embedding = get_multimodal_embedding(text=query_text)

# Calculate similarities
for item in database:
    sim = cosine_similarity(query_embedding, item['embedding'])
    print(f"ID: {item['id']} | Similarity: {sim:.4f}")

Filter Search Results by Similarity Threshold

In [ ]:
min_similarity_threshold = 0.32 # You can adjust this value

print(f"\nFiltering results for query: '{query_text}' with threshold >= {min_similarity_threshold:.2f}")

filtered_results = []
for item in database:
    sim = cosine_similarity(query_embedding, item['embedding'])
    if sim >= min_similarity_threshold:
        # Store the original item along with its similarity for easy access
        filtered_results.append({"item": item, "similarity": sim})

# Sort filtered results by similarity in descending order
filtered_results.sort(key=lambda x: x['similarity'], reverse=True)

if filtered_results:
    print("\nFiltered and Sorted Results:")
    for result in filtered_results:
        item = result['item']
        print(f"ID: {item['id']} | Similarity: {result['similarity']:.4f}")
        if item['type'] == 'text':
            print(f"Content: {item['content']}")
        elif item['type'] == 'image':
            print(f"Image URL: {item['url']}")
        print("-" * 30)
else:
    print("\nNo items found above the similarity threshold.")